# TP3 — PINN Resolution without additional data
## System of Poisson equations via PINNs

This notebook implements the resolution of **Test Problem 4 (TP4)** via **PINNs** without additional data.  
The objective is to exploit **Physics-Informed Neural Networks (PINNs)** and simulated IoT-like measurements.

This stage corresponds to the **Direct Problem Submodule** of the Inference Engine described in the paper.

In [ ]:
import sys
from pyprojroot import here

PROJECT_ROOT = str(here()) + "/"
sys.path.append(PROJECT_ROOT)
sys.dont_write_bytecode = True

In [ ]:
from paths import DRT_PATH, TP4_PATH, MODEL_PATH, COLUMN_PATH

ABS_PATH = PROJECT_ROOT + DRT_PATH + TP4_PATH
LOAD_MODEL = PROJECT_ROOT + MODEL_PATH + COLUMN_PATH

model_name = "column.blend"
load_data_bound = ABS_PATH + "./files/data.csv"
load_collocation_points = ABS_PATH + "files/input_int.csv"

fig_dir = "figures/"
loss_fig = ABS_PATH + fig_dir + "loss_nodata.png"

file_path = "column.xdmf"

params_save = ABS_PATH + "files/pred_parameters.csv"
model_name_features = "./models/file"
model_saved_name = './models/model_weights'

Import of packages

In [ ]:
import json
import torch
import pandas as pd
import fenics as fe
import matplotlib.pyplot as plt

from modelaquisition.msh2xdmf import Msh2Xdmf
from modelaquisition.bl2pina import Blend2Pina

from pina.operators import laplacian
from pina.condition import Condition
from pina.solvers.pinns import RBAPINN
from pina.problem import SpatialProblem
from pina.callbacks import MetricTracker
from pina.equation import SystemEquation
from pina.model import ResidualFeedForward
from pina import LabelTensor, Trainer, Plotter
from pytorch_lightning.callbacks import StochasticWeightAveraging

Set the double precision

In [ ]:
torch.set_default_dtype(torch.float64)

Load of the mesh points to make predictions through the trained network.

In [ ]:
mesh = fe.Mesh()
with fe.XDMFFile(file_path) as infile:
    infile.read(mesh)

mesh_points = LabelTensor(
    mesh.coordinates(),
    labels=['x', 'y', 'z']
)

Definition of useful variables

In [ ]:
collocation_points = 1_000
boundary_points = 500

# Network variables
lear_rate = 1e-3
decay_rt = 1e-8

# Solver variables
epochs = 5_000
acc_str = 'gpu'
batch_dim = None
num_layers = 2
num_neurons = 200
input_neurons = 3
output_neurons = 2

## Geometry management and loading of data

Here the notebook focuses on the interaction between the preprocessed Blender 3D model (See notebook *00_ProblemSettings.ipynb*) with the PINA Geometry module.
In addition. in this fase the simulated data is loaded.

In [ ]:
filename = LOAD_MODEL + model_name
column = Blend2Pina(filename)

column_int = column.intern()
column_bound = column.boundary()

In [ ]:
df = pd.read_csv(load_data_bound, sep=";", index_col=0)

df = df.sample(boundary_points)
input_pts = df.iloc[:, :3].values
input_pts = LabelTensor(
    x=torch.tensor(input_pts, dtype=torch.float64),
    labels=['x', 'y', 'z']
)

## Governing Physical Model

We consider a 3D domain $\Omega \subseteq \mathbb{R}^3$ representing a **column geometry**, with boundary $\Gamma = \partial \Omega$.  

The physical phenomenon is described by the following differenzial problem:

\begin{cases}
    \Delta u (x, y, z ) = F(x, y, z, u) & \Omega \\
    u (x, y, z) = B (x, y, z) & \partial \Omega
    \tag{1}
\end{cases}
where $u = (u_1, u_2)^T \in \mathbb{R}^2$,

\begin{equation}
    F (x, y, z, u) = \left(
        \begin{array}{c}
            2 u_1 \\
            2
        \end{array}
    \right), \qquad (x, y, z ) \in \Omega ,
    \tag{2}
\end{equation}

\begin{equation}
    B (x, y, z) = \left(
        \begin{array}{c}
            e^{x+y} \\
            x^2 - z
        \end{array}
    \right), \qquad (x, y, z ) \in \partial \Omega .
    \tag{2}
\end{equation}

In the next block, the problem is defined.

In [ ]:
class Poisson(SpatialProblem):
    input_variables = ['x','y','z']
    output_variables = ['u1', 'u2']
    spatial_domain =column_int

    @staticmethod
    def eq1(input_, output_):
        delta_u = laplacian(output_, input_, components=['u1'], d=['x','y','z'])
        force_term = 2 * output_.extract('u1')
        return delta_u - force_term
    
    @staticmethod
    def eq2(input_, output_):
        delta_u = laplacian(output_, input_, components=['u2'], d=['x','y','z'])
        force_term = 2
        return delta_u - force_term
    
    @staticmethod
    def bound1(input_, output_):
        return output_.extract('u1') - torch.exp((input_.extract('x') + input_.extract('y')))
    
    @staticmethod
    def bound2(input_, output_):
        return output_.extract('u2') - (input_.extract('x')**2 - input_.extract('z'))
    
    conditions = {
        "Omega" : Condition(
            location=column_int,
            equation=SystemEquation([eq1, eq2])
        ),
        "Gamma" : Condition(
            location=column_bound,
            equation=SystemEquation([bound1,bound2])
        )
    }

After the definition of the class, the sampling of collocation and boundary points is performed.

In [ ]:
problem = Poisson()

In [ ]:
try:
    df = pd.read_csv(load_collocation_points, sep=";", index_col=0)
    problem.discretise_domain(
        1,
        locations=["Omega", "Gamma"]
    )
    problem.input_pts["Omega"] = LabelTensor(
        torch.tensor(df.sample(collocation_points).values),
        labels=['x', 'y', 'z']
    )
    problem.input_pts["Gamma"] = input_pts

except:
    problem.discretise_domain(
        collocation_points,
        locations=["Omega", "Gamma"]
    )

    df = pd.DataFrame(
        problem.input_pts["Omega"].tensor.detach().numpy()
    )
    problem.input_pts["Gamma"] = input_pts

    df.to_csv(load_collocation_points, sep = ";")

## Physics-Informed Neural Network (PINN)

A Physics-Informed Neural Network is employed to approximate the solution

\begin{equation}
u(x,y,z) \approx u_{\theta}(x,y,z),
\end{equation}

where $ u_{\theta} $ is a neural network parameterized by weights $ \theta $.

In the inverse setting, the unknown physical parameters $ \boldsymbol{\mu} $ are treated as **trainable variables** and optimized jointly with the network weights.

### PINN Loss Function

The training of the PINN is driven by a composite loss function:

\begin{equation}
\mathcal{L} =
\mathcal{L}_{\text{PDE}} +
\mathcal{L}_{\text{BC}},
\end{equation}

where:

- **PDE residual loss**

\begin{equation}
\mathcal{L}_{\text{PDE}} =
\frac{1}{N_{\Omega}}
\sum_{i=1}^{N_{\Omega}}
\left\|
\Delta u_{\theta}(x_i) + (\alpha^2 + \beta^2)\pi^2 \lambda x_i \cos(\alpha \pi y_i)\sin(\beta \pi z_i)
\right\|^2,
\end{equation}

- **Boundary condition loss**

\begin{equation}
\mathcal{L}_{\text{BC}} =
\frac{1}{N_{\partial\Omega}}
\sum_{i=1}^{N_{\partial\Omega}}
\left\|
u_{\theta}(x_i) - \lambda x_i \cos(\alpha \pi y_i)\sin(\beta \pi z_i)
\right\|^2.
\end{equation}

Inizialization of the NN model, the PINN solver, the Trainer. Then, the training phase starts.

In [ ]:
# Model
model = ResidualFeedForward(input_dimensions=input_neurons,output_dimensions=output_neurons,n_layers=num_layers,inner_size=num_neurons)
# Solver
pinn=RBAPINN(
    problem=problem,
    model=model,
    optimizer_kwargs={
        'lr' : lear_rate,
        'weight_decay' : decay_rt
    },
)

# Trainer
trainer=Trainer(
    solver=pinn,
    max_epochs=epochs,
    batch_size=batch_dim,
    accelerator=acc_str,
    precision='64-true',
    callbacks=[MetricTracker(), StochasticWeightAveraging(swa_lrs=5e-4, swa_epoch_start=.9)]
)

# Training phase
trainer.train()

Plot of the losses related to the training process.

In [ ]:
my_pl = Plotter()
my_pl.plot_loss(
    trainer=trainer,
    metrics=['Omega_loss'],
    label='Omega',
    logy=True
)
my_pl.plot_loss(
    trainer=trainer,
    metrics=['Gamma_loss'],
    label='Gamma',
    logy=True
)

plt.savefig(loss_fig, transparent=True)
plt.show()

Definition of the function to implement the analytical solution

In [ ]:
def analytical(pts):
    u1 = torch.exp(pts.extract('x') + pts.extract('y'))
    u2 = pts.extract('x')**2 - pts.extract('z')
    sol = LabelTensor(
        torch.cat([u1, u2], axis=-1),
        labels=['u1', 'u2']
    )
    return sol

Elaboration of prediction via the PINN and of analytical solution on the training points.

In [ ]:
pred = pinn.neural_net(problem.input_pts["Omega"])
analit = analytical(problem.input_pts["Omega"])

err_matrix = LabelTensor(
    pred - analit.tensor,
    labels=['u1', 'u2']
)

Elaboration and print of errors on training points

In [ ]:
# Absolute errors
err_a_u1 = torch.norm(err_matrix.extract('u1'), p=2).tensor.item()
err_a_u2 = torch.norm(err_matrix.extract('u2'), p=2).tensor.item()

# Relative Errors
err_r_u1 = err_a_u1 / torch.norm(analit.extract('u1'), p=2).tensor.item()
err_r_u2 = err_a_u2 / torch.norm(analit.extract('u2'), p=2).tensor.item()

print("TRAINING ERRORS")
print("ABSOLUTE ERRORS")
print(f"u_1 -> {err_a_u1:.4e}")
print(f"u_2 -> {err_a_u2:.4e}")
print("----------------------------")
print("RELATIVE ERRORS")
print(f"u_1 -> {err_r_u1:.4e}")
print(f"u_2 -> {err_r_u2:.4e}")

In [ ]:
print("ERROR ON MAGNITUDE")
err = torch.norm(pred.tensor - analit.tensor).item()
err_rel = err / torch.norm(analit.tensor).item()

print(f"Absolute error: {err:.4e}")
print(f"Relative error: {err_rel:.4e}")

Predictions and analytical solution on mesh points.

In [ ]:
pred = pinn.neural_net(mesh_points)
analit = analytical(mesh_points)

err_matrix = LabelTensor(
    torch.abs(pred - analit.tensor),
    labels=['u1', 'u2']
)

Elaboration and print of errors on testing points.

In [ ]:
# Absolute errors
err_a_u1 = torch.norm(err_matrix.extract('u1'), p=2).tensor.item()
err_a_u2 = torch.norm(err_matrix.extract('u2'), p=2).tensor.item()

# Relative Errors
err_r_u1 = err_a_u1 / torch.norm(analit.extract('u1'), p=2).tensor.item()
err_r_u2 = err_a_u2 / torch.norm(analit.extract('u2'), p=2).tensor.item()

print("TESTING ERRORS")
print("ABSOLUTE ERRORS")
print(f"u_1 -> {err_a_u1:.4e}")
print(f"u_2 -> {err_a_u2:.4e}")
print("----------------------------")
print("RELATIVE ERRORS")
print(f"u_1 -> {err_r_u1:.4e}")
print(f"u_2 -> {err_r_u2:.4e}")

In [ ]:
print("ERROR ON MAGNITUDE - TESTING")
err_t = torch.norm(pred.tensor - analit.tensor).item()
err_rel_t = err_t / torch.norm(analit.tensor).item()

print(f"Absolute error: {err_t:.4e}")
print(f"Relative error: {err_rel_t:.4e}")

## Export of solutions and errors and saving model

In [ ]:
column_xdmf = Msh2Xdmf("column.msh", "column")

column_xdmf.reset_files("_nodata_sol_pinn")
column_xdmf.reset_files("_nodata_sol_an")
column_xdmf.reset_files("_nodata_err_r")

first_column = err_matrix.extract('u1') / torch.norm(analit.extract('u1').tensor, p=2).item()
second_column = err_matrix.extract('u2') / torch.norm(analit.extract('u2').tensor, p=2).item()

err_r_matrix = torch.cat([first_column, second_column], axis=-1)

column_xdmf.add_solution(pred.tensor.detach().numpy(), "_nodata_sol_pinn", "solution_pinn")
column_xdmf.add_solution(analit.tensor.detach().numpy(), "_nodata_sol_an", "solution_an")
column_xdmf.add_solution(err_r_matrix.detach().numpy(), "_nodata_err_r", "error")

In [ ]:
torch.save(model.state_dict(), model_saved_name + "_nodata" + '.pth')


In [ ]:
create_model = {
    'num_hidden_layers' : num_layers,
    'num_hidden_neurons' : num_neurons,
    'input_dim' : input_neurons,
    'output_dim' : output_neurons
}

with open(model_name_features + "_nodata" + '.json', 'w') as file:
    json.dump(create_model, file, indent=4)